# Data provenance with a bigger dataframe

In [1]:
import $ivy.`org.apache.spark::spark-sql:4.1.1`

import $ivy.$

In [2]:
val version = scala.io.Source.fromFile("../VERSION")  // Get version from file
  .getLines().next().trim

interp.load.ivy("org.dataprov.dp" %% "dp-spark" % version)  // use porogrammatic API 

// // For publishLocal (~/.ivy2/local)
// import $ivy.`org.dataprov.dp::dp-spark:0.0.1` // N.B: version must be specified explicitly with this method

// // For publishM2 (~/.m2)
// import $repo.`file:///home/ronan/.m2/repository`
// import $ivy.`org.dataprov.dp::dp-spark:0.0.1` // N.B: version must be specified explicitly with this method

version: String = "0.0.1"

In [3]:
import java.sql.Date

import org.apache.spark.sql.{SparkSession, DataFrame}
import org.apache.spark.sql.catalyst.plans.logical.LogicalPlan
import org.apache.spark.sql.execution.SparkPlan
import org.apache.spark.sql.functions._

import org.dataprov.dp.sparkdataprovenance.ProvenanceApi._
import org.dataprov.dp.sparkdataprovenance.LogicalPlanWithProvenance
import org.dataprov.dp.sparkdataprovenance.SparkProvenanceExtension
import org.dataprov.dp.sparkdataprovenance.FullWhyProvenanceBuilder
import org.dataprov.dp.sparkdataprovenance.SemiWhyProvenanceBuilder

import java.sql.Date
import org.apache.spark.sql.{SparkSession, DataFrame}
import org.apache.spark.sql.catalyst.plans.logical.LogicalPlan
import org.apache.spark.sql.execution.SparkPlan
import org.apache.spark.sql.functions._
import org.dataprov.dp.sparkdataprovenance.ProvenanceApi._
import org.dataprov.dp.sparkdataprovenance.LogicalPlanWithProvenance
import org.dataprov.dp.sparkdataprovenance.SparkProvenanceExtension
import org.dataprov.dp.sparkdataprovenance.FullWhyProvenanceBuilder
import org.dataprov.dp.sparkdataprovenance.SemiWhyProvenanceBuilder

## Initilize Spark Session

In [4]:
import org.apache.spark.sql.SparkSession

val sparkWhy = SparkSession.builder()
    .appName("notebook-demo-why-provenance")
    .master("local[*]")
    // Vos extensions de provenance
    .withExtensions(
        new SparkProvenanceExtension(
            provenanceBuilder = SemiWhyProvenanceBuilder
        )
    )
    .config("spark.provenance.enabled", "true")
    .getOrCreate()

println(s"Spark provenance enabled: ${sparkWhy.conf.get("spark.provenance.enabled")}")
sparkWhy.sparkContext.setLogLevel("ERROR")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/18 11:49:22 INFO SparkContext: Running Spark version 4.1.1
26/06/18 11:49:22 INFO SparkContext: OS info Mac OS X, 26.4.1, aarch64
26/06/18 11:49:22 INFO SparkContext: Java version 17.0.10+7
26/06/18 11:49:22 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/18 11:49:22 INFO ResourceUtils: ==============================================================
26/06/18 11:49:22 INFO ResourceUtils: No custom resources configured for spark.driver.
26/06/18 11:49:22 INFO ResourceUtils: ==============================================================
26/06/18 11:49:22 INFO SparkContext: Submitted application: notebook-demo-why-provenance
26/06/18 11:49:22 INFO SecurityManager: Changing view acls to: mac-ABALLA16
26/06/18 11:49:22 INFO SecurityManager: Changing modify acls to: mac-ABALLA16
26/06/18 11:49:22 INFO SecurityManager: Changing

Spark provenance enabled: true


import org.apache.spark.sql.SparkSession
sparkWhy: SparkSession = org.apache.spark.sql.classic.SparkSession@14c83eb7

## Loading file and adding provenance

In [5]:
// Example of loading a Parquet file and adding provenance

val localPath = "file:///Users/mac-ABALLA16/Downloads/product_hierarchy"
val df = sparkWhy.read.format("parquet").load(localPath)

println(s"Nombre de lignes réelles chargées : ${df.count()}")

val dfWithProv = df.addProvenanceColumn(col("model_code"))
dfWithProv.show(10, truncate = false)

// val truc = dfWithProv.select("model_code").distinct()
// truc.show(10, false)

Nombre de lignes réelles chargées : 55526486
+----------+-----------+---------------------------+-------------------+------------------------------------+---------------+-----------------------+-------------+-----------------------------------+--------------+-------------------+-----------------+---------------------------+---------------+
|model_code|family_code|family_name                |sub_department_code|sub_department_name                 |department_code|department_name        |universe_code|universe_name                      |hierarchy_type|validity_start_date|validity_end_date|tech_ingestion_datetime_utc|_provenance_tag|
+----------+-----------+---------------------------+-------------------+------------------------------------+---------------+-----------------------+-------------+-----------------------------------+--------------+-------------------+-----------------+---------------------------+---------------+
|1         |5053       |UNDERWEAR TEAM SPORT SENIOR|1970        

localPath: String = "file:///Users/mac-ABALLA16/Downloads/product_hierarchy"
df: DataFrame = [model_code: bigint, family_code: bigint ... 11 more fields]
dfWithProv: DataFrame = [model_code: bigint, family_code: bigint ... 12 more fields]

In [6]:
val dfLittle = dfWithProv.select("model_code", "family_name", "department_name","sub_department_name")
dfLittle.show(10, false)

+----------+---------------------------+-----------------------+------------------------------------+---------------+
|model_code|family_name                |department_name        |sub_department_name                 |_provenance_tag|
+----------+---------------------------+-----------------------+------------------------------------+---------------+
|1         |UNDERWEAR TEAM SPORT SENIOR|Football/soccer        |TEAMSPORT THERMICAL APPAREL         |1              |
|2         |UNDERWEAR TEAM SPORT SENIOR|SOCCER / FUTSAL / SEPAK|MAN FOOTBALL                        |2              |
|2         |UNDERWEAR TEAM SPORT SENIOR|FOOTBALL & FUTSAL      |MEN FOOTBALL                        |2              |
|2         |UNDERWEAR TEAM SPORT SENIOR|Football / Soccer      |Training and accessories            |2              |
|4         |Ad foot underwear          |Football / Soccer      |Adult football                      |4              |
|4         |UNDERWEAR TEAM SPORT SENIOR|Football/soccer 

dfLittle: DataFrame = [model_code: bigint, family_name: string ... 3 more fields]

In [7]:
val distinct = dfLittle.select("model_code").distinct()
distinct.show(false)

+----------+---------------+
|model_code|_provenance_tag|
+----------+---------------+
|2700      |[2700]         |
|15594     |[15594]        |
|18402     |[18402]        |
|19891     |[19891]        |
|66563     |[66563]        |
|69430     |[69430]        |
|69546     |[69546]        |
|70749     |[70749]        |
|70757     |[70757]        |
|70811     |[70811]        |
|72784     |[72784]        |
|83956     |[83956]        |
|102179    |[102179]       |
|102209    |[102209]       |
|102454    |[102454]       |
|119683    |[119683]       |
|164107    |[164107]       |
|228563    |[228563]       |
|254246    |[254246]       |
|278021    |[278021]       |
+----------+---------------+
only showing top 20 rows


distinct: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [model_code: bigint, _provenance_tag: array<string>]

In [8]:
val aggregate = dfLittle.groupBy("sub_department_name").agg(count("*").as("count"))
aggregate.show(10,false)

+--------------------------------+------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

aggregate: DataFrame = [sub_department_name: string, count: bigint ... 1 more field]

In [9]:
val dfLittle2 = dfWithProv.select("model_code", "universe_code", "hierarchy_type")

dfLittle2: DataFrame = [model_code: bigint, universe_code: bigint ... 2 more fields]

In [10]:
val join = dfLittle.join(dfLittle2, "model_code")
join.show(30, false)

val joinThenDistinct = join.select("model_code", "family_name").distinct()
//joinThenDistinct.show(10, false)

+----------+---------------+---------------+-------------------------+-------------+--------------+---------------+
|model_code|family_name    |department_name|sub_department_name      |universe_code|hierarchy_type|_provenance_tag|
+----------+---------------+---------------+-------------------------+-------------+--------------+---------------+
|66563     |MAP BOOK HIKING|HIKING         |BACKPACK EQUIPMENT HIKING|5            |DMI           |[66563]        |
|66563     |MAP BOOK HIKING|HIKING         |BACKPACK EQUIPMENT HIKING|5            |RETAIL        |[66563]        |
|66563     |MAP BOOK HIKING|HIKING         |BACKPACK EQUIPMENT HIKING|5            |RETAIL        |[66563]        |
|66563     |MAP BOOK HIKING|HIKING         |BACKPACK EQUIPMENT HIKING|5            |DMI           |[66563]        |
|66563     |MAP BOOK HIKING|HIKING         |BACKPACK EQUIPMENT HIKING|9            |RETAIL        |[66563]        |
|66563     |MAP BOOK HIKING|HIKING         |BACKPACK EQUIPMENT HIKING|46

join: DataFrame = [model_code: bigint, family_name: string ... 5 more fields]
joinThenDistinct: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [model_code: bigint, family_name: string ... 1 more field]

In [11]:

val finalRows = joinThenDistinct.select("_provenance_tag").filter(col("model_code") === "1037865" || col("model_code") === "1127317")
finalRows.show(20, false)

//sparkWhy.conf.set("spark.provenance.enabled", "false")

withProvenanceDisabled(sparkWhy) {
  val finalTags = finalRows
    .select(explode((col("_provenance_tag"))).as("_provenance_tag"))
    .distinct()

   val initialRows = dfWithProv
    .join(finalTags, Seq("_provenance_tag"), "left_semi")

   initialRows.show(20, false)
}


+---------------+
|_provenance_tag|
+---------------+
|[1037865]      |
|[1037865]      |
|[1127317]      |
|[1127317]      |
|[1127317]      |
|[1127317]      |
+---------------+

+---------------+----------+-----------+-------------------------------+-------------------+------------------------------------+---------------+----------------------------------+-------------+----------------------------------+--------------+-------------------+-----------------+---------------------------+
|_provenance_tag|model_code|family_code|family_name                    |sub_department_code|sub_department_name                 |department_code|department_name                   |universe_code|universe_name                     |hierarchy_type|validity_start_date|validity_end_date|tech_ingestion_datetime_utc|
+---------------+----------+-----------+-------------------------------+-------------------+------------------------------------+---------------+----------------------------------+-------------+---

finalRows: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [_provenance_tag: array<string>]

In [12]:
val df = sparkWhy.read.format("parquet").load(localPath)
println(s"Nombre de lignes réelles chargées : ${df.count()}")

val dfWithProv = df.addProvenanceColumn
dfWithProv.show(10, false)


Nombre de lignes réelles chargées : 55526486
+----------+-----------+---------------------------+-------------------+------------------------------------+---------------+-----------------------+-------------+-----------------------------------+--------------+-------------------+-----------------+---------------------------+------------------------------------+
|model_code|family_code|family_name                |sub_department_code|sub_department_name                 |department_code|department_name        |universe_code|universe_name                      |hierarchy_type|validity_start_date|validity_end_date|tech_ingestion_datetime_utc|_provenance_tag                     |
+----------+-----------+---------------------------+-------------------+------------------------------------+---------------+-----------------------+-------------+-----------------------------------+--------------+-------------------+-----------------+---------------------------+------------------------------------+
|

df: DataFrame = [model_code: bigint, family_code: bigint ... 11 more fields]
dfWithProv: DataFrame = [model_code: bigint, family_code: bigint ... 12 more fields]

In [13]:
val dfLittle = dfWithProv.select("model_code", "family_name", "department_name","universe_name")
dfLittle.show(10, false)

+----------+---------------------------+-----------------------+-----------------------------------+------------------------------------+
|model_code|family_name                |department_name        |universe_name                      |_provenance_tag                     |
+----------+---------------------------+-----------------------+-----------------------------------+------------------------------------+
|1         |UNDERWEAR TEAM SPORT SENIOR|Football/soccer        |Football univ                      |99481756-e3e2-45af-b173-e77dade97a20|
|2         |UNDERWEAR TEAM SPORT SENIOR|SOCCER / FUTSAL / SEPAK|TEAMSPORTS UNIVERS                 |37de0d66-432a-4959-b087-de8ae8c0b53f|
|2         |UNDERWEAR TEAM SPORT SENIOR|FOOTBALL & FUTSAL      |TEAMSPORTS                         |4d29c5ef-b053-4711-8916-473cd223f91a|
|2         |UNDERWEAR TEAM SPORT SENIOR|Football / Soccer      |Teamsports                         |78ff45c8-7878-477e-a86f-964f5e630cb0|
|4         |Ad foot underwear     

dfLittle: DataFrame = [model_code: bigint, family_name: string ... 3 more fields]

In [14]:
val distinct = dfWithProv.select("family_name").distinct()
distinct.show(false)

+--------------------------------------+--------------------------------------+
|family_name                           |_provenance_tag                       |
+--------------------------------------+--------------------------------------+
|'PULL-APART' POLE FISHING RODS, ACCESS|[44e3bb1f-a9cd-42f4-8774-357a94d2f280]|
|(!) EMPTY EXPO BACKBOARDS             |[bc1e508b-83c0-438f-a238-5147ea6a2151]|
|(!) EXPO BACKBOARDS                   |[b28b5924-fabe-4765-bda8-f9724d15fe10]|
|0L Replica                            |[4ab8256c-e370-408b-8f9f-624c838acb04]|
|10L TO 30L NATURE HIKING BACKPACKS    |[d7210261-ae68-48bc-a9ef-7a76aec69b8d]|
|11 FOOTBALL BALLS                     |[d82d95b1-3ba9-4898-89a4-c8ed2afafb6f]|
|12"/14" Bikes (3-5 years)             |[574fde9f-0724-465c-80af-5de39db0e6ee]|
|16 INCHES BIKES (4-6 YEARS)           |[ca8ba839-fce8-4f0e-b762-39bdda886580]|
|16I                                   |[eb777f08-80ea-4b2e-af1e-79491c7bfa14]|
|20 GA CARTRIDGES/SLUGS                |

distinct: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [family_name: string, _provenance_tag: array<string>]

In [15]:
// This query is counting the number of rows for each family_name where model_code is less than or equal to 5
val aggregate = dfWithProv.filter(col("model_code") <= 20).groupBy("family_name").agg(count("*").as("count"))
aggregate.show(10,false)

+-----------------------------+-----+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

aggregate: DataFrame = [family_name: string, count: bigint ... 1 more field]